# Lab 08 Practice: Neural Networks
**CS201L: Artificial Intelligence Laboratory**  
**Indian Institute of Technology, Dharwad**  
**Date Fruit Dataset — Multi-class Classification**

## Imports & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

torch.manual_seed(42)
np.random.seed(42)

print('PyTorch version:', torch.__version__)

---
## 1. Load Data

In [ ]:
# ── Adjust paths if your CSVs are in a sub-folder ──
DATA_DIR = 'data'   # change to '.' if CSVs are in the same folder

train_df = pd.read_csv(os.path.join(DATA_DIR, 'DateFruit_Train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'DateFruit_Validation.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'DateFruit_Test.csv'))

print('Train shape    :', train_df.shape)
print('Validation shape:', val_df.shape)
print('Test shape     :', test_df.shape)
print('Classes        :', train_df['Class'].unique())

## 2. Encode Labels & Build Tensors

In [ ]:
le = LabelEncoder()
le.fit(train_df['Class'])

def df_to_tensors(df, le):
    """Split a DataFrame into (X tensor, y tensor)."""
    X = torch.tensor(df.drop(columns=['Class']).values, dtype=torch.float32)
    y = torch.tensor(le.transform(df['Class']), dtype=torch.long)
    return X, y

X_train_raw, y_train = df_to_tensors(train_df, le)
X_val_raw,   y_val   = df_to_tensors(val_df,   le)
X_test_raw,  y_test  = df_to_tensors(test_df,  le)

INPUT_SIZE  = X_train_raw.shape[1]   # 34
OUTPUT_SIZE = len(le.classes_)       # 7
print(f'Input size: {INPUT_SIZE}, Output size: {OUTPUT_SIZE}')

---
## 3. Neural Network Definitions
### 3.1 Single Hidden Layer

In [ ]:
class SingleHiddenNet(nn.Module):
    """Input → Linear → Tanh → Output (Softmax handled by CrossEntropyLoss)."""

    def __init__(self, input_size, h1, output_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, h1),
            nn.Tanh(),
            nn.Linear(h1, output_size)
            # Softmax is implicit in nn.CrossEntropyLoss
        )

    def forward(self, x):
        return self.layers(x)

### 3.2 Two Hidden Layers

In [ ]:
class TwoHiddenNet(nn.Module):
    """Input → Linear → Tanh → Linear → Tanh → Output."""

    def __init__(self, input_size, h1, h2, output_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, h1),
            nn.Tanh(),
            nn.Linear(h1, h2),
            nn.Tanh(),
            nn.Linear(h2, output_size)
        )

    def forward(self, x):
        return self.layers(x)

---
## 4. Training & Evaluation Utilities

In [ ]:
def train_model(model, X_tr, y_tr, X_val, y_val,
                lr=0.01, max_epochs=1000, patience=None, threshold=1e-3):
    """
    Train `model` with SGD + CrossEntropyLoss.
    Returns (epoch_losses_train, best_val_accuracy).
    """
    if patience is None:
        patience = max(10, int(0.1 * max_epochs))

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    epoch_losses = []
    prev_loss    = float('inf')
    counter      = 0

    for epoch in range(1, max_epochs + 1):
        # ── Training step ──────────────────────────────────────
        model.train()
        optimizer.zero_grad()
        outputs    = model(X_tr)
        train_loss = criterion(outputs, y_tr)
        train_loss.backward()
        optimizer.step()

        loss_val = train_loss.item()
        epoch_losses.append(loss_val)

        # ── Convergence check ──────────────────────────────────
        if abs(loss_val - prev_loss) < threshold:
            counter += 1
        else:
            counter = 0
        prev_loss = loss_val

        if counter >= patience:
            print(f'  Converged at epoch {epoch}')
            break

    # ── Validation accuracy ────────────────────────────────────
    val_acc = evaluate_accuracy(model, X_val, y_val)
    return epoch_losses, val_acc


def evaluate_accuracy(model, X, y):
    """Return accuracy (float) on dataset (X, y)."""
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        preds   = torch.argmax(outputs, dim=1)
    return (preds == y).float().mean().item()


def get_predictions(model, X):
    """Return numpy array of predicted class indices."""
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        preds   = torch.argmax(outputs, dim=1)
    return preds.numpy()


def print_metrics(y_true, y_pred, label=''):
    """Print confusion matrix + all required metrics."""
    print(f'\n========== {label} ==========')
    print('Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))
    print(f'Accuracy          : {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision (Macro) : {precision_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
    print(f'Precision (Micro) : {precision_score(y_true, y_pred, average="micro", zero_division=0):.4f}')
    print(f'Recall    (Macro) : {recall_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
    print(f'Recall    (Micro) : {recall_score(y_true, y_pred, average="micro", zero_division=0):.4f}')
    print(f'F1-Score  (Macro) : {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
    print(f'F1-Score  (Micro) : {f1_score(y_true, y_pred, average="micro", zero_division=0):.4f}')


def plot_losses(loss_dict, title='Epoch vs Training Loss'):
    """Plot training loss curves for multiple architectures."""
    plt.figure(figsize=(9, 5))
    for arch_name, losses in loss_dict.items():
        plt.plot(losses, label=arch_name)
    plt.xlabel('Epoch')
    plt.ylabel('Cross-Entropy Loss')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def save_models(model_dict, save_dir='saved_models'):
    """Save all models in model_dict to `save_dir`."""
    os.makedirs(save_dir, exist_ok=True)
    for name, model in model_dict.items():
        path = os.path.join(save_dir, f'{name}.pth')
        torch.save(model.state_dict(), path)
        print(f'Saved: {path}')

---
## Part A — Original (Un-normalised) Data
### A1. Single Hidden Layer Architectures

In [ ]:
# Hyperparameters
LR         = 0.01
MAX_EPOCHS = 1000

single_configs_A = [(34,), (64,)]   # (h1,)
single_models_A  = {}
single_losses_A  = {}
single_val_acc_A = {}

for (h1,) in single_configs_A:
    name = f'1HL_h{h1}'
    print(f'\nTraining {name} ...')
    model = SingleHiddenNet(INPUT_SIZE, h1, OUTPUT_SIZE)
    losses, val_acc = train_model(
        model, X_train_raw, y_train, X_val_raw, y_val,
        lr=LR, max_epochs=MAX_EPOCHS
    )
    single_models_A[name]  = model
    single_losses_A[name]  = losses
    single_val_acc_A[name] = val_acc
    print(f'  Validation Accuracy: {val_acc:.4f}')

plot_losses(single_losses_A, title='Part A — Single Hidden Layer: Epoch vs Loss')

### A2. Two Hidden Layer Architectures

In [ ]:
two_configs_A = [(34, 34), (64, 16)]
two_models_A  = {}
two_losses_A  = {}
two_val_acc_A = {}

for (h1, h2) in two_configs_A:
    name = f'2HL_h{h1}_h{h2}'
    print(f'\nTraining {name} ...')
    model = TwoHiddenNet(INPUT_SIZE, h1, h2, OUTPUT_SIZE)
    losses, val_acc = train_model(
        model, X_train_raw, y_train, X_val_raw, y_val,
        lr=LR, max_epochs=MAX_EPOCHS
    )
    two_models_A[name]  = model
    two_losses_A[name]  = losses
    two_val_acc_A[name] = val_acc
    print(f'  Validation Accuracy: {val_acc:.4f}')

plot_losses(two_losses_A, title='Part A — Two Hidden Layers: Epoch vs Loss')

### A3. Select Best Architectures & Evaluate on Test Set

In [ ]:
# Pick best from each group (highest validation accuracy)
best_single_A = max(single_val_acc_A, key=single_val_acc_A.get)
best_two_A    = max(two_val_acc_A,    key=two_val_acc_A.get)

print('Part A — Best single hidden layer :', best_single_A,
      f'(val acc = {single_val_acc_A[best_single_A]:.4f})')
print('Part A — Best two hidden layers   :', best_two_A,
      f'(val acc = {two_val_acc_A[best_two_A]:.4f})')

y_test_np = y_test.numpy()

# Single hidden layer — test metrics
y_pred_single_A = get_predictions(single_models_A[best_single_A], X_test_raw)
print_metrics(y_test_np, y_pred_single_A,
              label=f'Part A | {best_single_A} | Test')

# Two hidden layers — test metrics
y_pred_two_A = get_predictions(two_models_A[best_two_A], X_test_raw)
print_metrics(y_test_np, y_pred_two_A,
              label=f'Part A | {best_two_A} | Test')

### A4. Save Part A Models

In [ ]:
all_models_A = {**single_models_A, **two_models_A}
save_models(all_models_A, save_dir='saved_models/partA')

---
## Part B — Z-Score Normalised Data
### B1. Compute Z-Score Normalisation (from training statistics only)

In [ ]:
# Compute mean and std from TRAINING data only
mean_tr = X_train_raw.mean(dim=0)   # shape: (34,)
std_tr  = X_train_raw.std(dim=0)    # shape: (34,)

# Avoid division by zero for constant features
std_tr = std_tr.clamp(min=1e-8)

# Apply z-score to all splits
X_train_norm = (X_train_raw - mean_tr) / std_tr
X_val_norm   = (X_val_raw   - mean_tr) / std_tr
X_test_norm  = (X_test_raw  - mean_tr) / std_tr

print('Normalised X_train — mean (should be ≈0):', X_train_norm.mean(dim=0).abs().max().item())
print('Normalised X_train — std  (should be ≈1):', X_train_norm.std(dim=0).mean().item())

### B2. Single Hidden Layer on Normalised Data

In [ ]:
single_configs_B = [(34,), (64,)]
single_models_B  = {}
single_losses_B  = {}
single_val_acc_B = {}

for (h1,) in single_configs_B:
    name = f'1HL_h{h1}'
    print(f'\nTraining {name} (normalised) ...')
    model = SingleHiddenNet(INPUT_SIZE, h1, OUTPUT_SIZE)
    losses, val_acc = train_model(
        model, X_train_norm, y_train, X_val_norm, y_val,
        lr=LR, max_epochs=MAX_EPOCHS
    )
    single_models_B[name]  = model
    single_losses_B[name]  = losses
    single_val_acc_B[name] = val_acc
    print(f'  Validation Accuracy: {val_acc:.4f}')

plot_losses(single_losses_B, title='Part B — Single Hidden Layer (Normalised): Epoch vs Loss')

### B3. Two Hidden Layers on Normalised Data

In [ ]:
two_configs_B = [(34, 34), (64, 16)]
two_models_B  = {}
two_losses_B  = {}
two_val_acc_B = {}

for (h1, h2) in two_configs_B:
    name = f'2HL_h{h1}_h{h2}'
    print(f'\nTraining {name} (normalised) ...')
    model = TwoHiddenNet(INPUT_SIZE, h1, h2, OUTPUT_SIZE)
    losses, val_acc = train_model(
        model, X_train_norm, y_train, X_val_norm, y_val,
        lr=LR, max_epochs=MAX_EPOCHS
    )
    two_models_B[name]  = model
    two_losses_B[name]  = losses
    two_val_acc_B[name] = val_acc
    print(f'  Validation Accuracy: {val_acc:.4f}')

plot_losses(two_losses_B, title='Part B — Two Hidden Layers (Normalised): Epoch vs Loss')

### B4. Select Best Architectures & Evaluate on Test Set

In [ ]:
best_single_B = max(single_val_acc_B, key=single_val_acc_B.get)
best_two_B    = max(two_val_acc_B,    key=two_val_acc_B.get)

print('Part B — Best single hidden layer :', best_single_B,
      f'(val acc = {single_val_acc_B[best_single_B]:.4f})')
print('Part B — Best two hidden layers   :', best_two_B,
      f'(val acc = {two_val_acc_B[best_two_B]:.4f})')

# Single hidden layer — test metrics (normalised)
y_pred_single_B = get_predictions(single_models_B[best_single_B], X_test_norm)
print_metrics(y_test_np, y_pred_single_B,
              label=f'Part B | {best_single_B} | Test (normalised)')

# Two hidden layers — test metrics (normalised)
y_pred_two_B = get_predictions(two_models_B[best_two_B], X_test_norm)
print_metrics(y_test_np, y_pred_two_B,
              label=f'Part B | {best_two_B} | Test (normalised)')

### B5. Save Part B Models

In [ ]:
all_models_B = {**single_models_B, **two_models_B}
save_models(all_models_B, save_dir='saved_models/partB')

---
## Summary — Validation Accuracies Across All Architectures

In [ ]:
summary = pd.DataFrame({
    'Architecture': list(single_val_acc_A) + list(two_val_acc_A) +
                    list(single_val_acc_B) + list(two_val_acc_B),
    'Part': (['A-Single']*2 + ['A-Two']*2 +
             ['B-Single']*2 + ['B-Two']*2),
    'Val Accuracy': (list(single_val_acc_A.values()) + list(two_val_acc_A.values()) +
                     list(single_val_acc_B.values()) + list(two_val_acc_B.values()))
})

summary['Val Accuracy'] = summary['Val Accuracy'].round(4)
print(summary.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue']*2 + ['tomato']*2 + ['seagreen']*2 + ['goldenrod']*2
bars = ax.bar(range(len(summary)), summary['Val Accuracy'], color=colors)
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(
    [f"{r['Part']}\n{r['Architecture']}" for _, r in summary.iterrows()],
    fontsize=9
)
ax.set_ylabel('Validation Accuracy')
ax.set_title('Summary: Validation Accuracy — All Architectures')
ax.set_ylim(0, 1.05)
for bar, val in zip(bars, summary['Val Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()